In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name != "ecohome_solution":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

Project root: c:\Users\praveen.vanaparthy\Downloads\New folder\ecohome_solution
Python: c:\Users\praveen.vanaparthy\Downloads\New folder\ecohome_solution\.venv\Scripts\python.exe


In [2]:
from models.energy import Base, engine

Base.metadata.create_all(engine)

print("Database tables created successfully.")

Database tables created successfully.


In [3]:
from datetime import datetime, timedelta
from sqlalchemy.orm import Session

from models.energy import EnergyUsage, SolarGeneration, engine


devices = {
    "EV": 7.2,
    "HVAC": 3.5,
    "Dishwasher": 1.5,
    "Washing Machine": 1.2,
    "Pool Pump": 2.0,
    "Water Heater": 4.0,
}


with Session(engine) as session:

    # Avoid duplicate sample data if notebook is run again
    session.query(EnergyUsage).delete()
    session.query(SolarGeneration).delete()

    start = datetime.now().replace(
        hour=0,
        minute=0,
        second=0,
        microsecond=0
    )

    for day in range(7):

        for hour in range(24):

            timestamp = start - timedelta(days=6-day, hours=23-hour)

            # Different consumption patterns by time
            if 0 <= hour < 6:
                period = "off_peak"
                price = 0.08
            elif 6 <= hour < 17:
                period = "standard"
                price = 0.14
            elif 17 <= hour < 21:
                period = "peak"
                price = 0.25
            else:
                period = "standard"
                price = 0.14

            for device, base_usage in devices.items():

                # Device-specific usage pattern
                if device == "EV":
                    usage = base_usage if 0 <= hour < 6 else 0.0

                elif device == "HVAC":
                    usage = base_usage * (
                        1.2 if 12 <= hour <= 18 else 0.7
                    )

                elif device == "Dishwasher":
                    usage = base_usage if hour == 20 else 0.0

                elif device == "Washing Machine":
                    usage = base_usage if hour == 19 else 0.0

                elif device == "Pool Pump":
                    usage = base_usage if 12 <= hour < 16 else 0.0

                elif device == "Water Heater":
                    usage = base_usage * (
                        1.0 if 6 <= hour <= 8 else 0.3
                    )

                else:
                    usage = base_usage * 0.5

                if usage > 0:
                    session.add(
                        EnergyUsage(
                            timestamp=timestamp,
                            device_type=device,
                            energy_kwh=round(usage, 2),
                            cost=round(usage * price, 2),
                            usage_period=period,
                        )
                    )

            # Solar generation
            if 7 <= hour <= 17:

                solar = max(
                    0,
                    5.5 * (1 - abs(12 - hour) / 6)
                )

                weather = (
                    "sunny"
                    if solar > 3.5
                    else "partly_cloudy"
                )

            else:
                solar = 0
                weather = "night"

            session.add(
                SolarGeneration(
                    timestamp=timestamp,
                    generation_kwh=round(solar, 2),
                    weather_condition=weather,
                )
            )

    session.commit()

print("Sample energy and solar data inserted successfully.")

Sample energy and solar data inserted successfully.


In [4]:
from sqlalchemy import inspect, select, func
from models.energy import EnergyUsage, SolarGeneration, engine

inspector = inspect(engine)

print("Tables:")
print(inspector.get_table_names())

with Session(engine) as session:

    energy_count = session.scalar(
        select(func.count()).select_from(EnergyUsage)
    )

    solar_count = session.scalar(
        select(func.count()).select_from(SolarGeneration)
    )

print(f"\nEnergyUsage records: {energy_count}")
print(f"SolarGeneration records: {solar_count}")

Tables:
['energy_usage', 'solar_generation']

EnergyUsage records: 420
SolarGeneration records: 168
